# Model Explorer

Load a trained model from `models/{run_id}/` and run it on train/validation data.

**Usage:** Set `RUN_ID` below to the run ID you want to load (e.g. `"mlx_smoke_20260324_143052"`).
The notebook will:
1. Extract the model definition from `models/{run_id}/log.txt` (which contains the training script)
2. Load weights from the quantized `model.int8.ptz` checkpoint (or raw `model.npz`)
3. Load train and validation data using baseline data loading utilities
4. Let you run the model on examples and inspect predictions

In [ ]:
# === CONFIGURATION ===
# Set this to the run ID of the model you want to load.
# Available runs can be found with: !ls models/
RUN_ID = "mlx_smoke"

# Whether to load the quantized (int8) or raw (fp32) checkpoint
USE_QUANTIZED = True

# Sequence length for data batches
SEQ_LEN = 1024

# Data paths (defaults match the baseline)
DATA_PATH = "./data/datasets/fineweb10B_sp1024"
TOKENIZER_PATH = "./data/tokenizers/fineweb_1024_bpe.model"

In [ ]:
import glob
import importlib.util
import os
import pickle
import sys
import tempfile
import zlib
from pathlib import Path

import numpy as np
import sentencepiece as spm

import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten, tree_unflatten

## 1. Load the training script from the log file

The `.txt` log files contain the full training script followed by training output.
We extract just the Python source, write it to a temp file, and import it as a module.
This gives us the exact model class and hyperparameters that were used for training.

In [ ]:
run_dir = Path("models") / RUN_ID
log_txt = run_dir / "log.txt"
assert log_txt.exists(), f"Log file not found: {log_txt}"

# Determine checkpoint path
if USE_QUANTIZED:
    ckpt_path = run_dir / "model.int8.ptz"
else:
    ckpt_path = run_dir / "model.npz"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

print(f"Run dir:    {run_dir}")
print(f"Log file:   {log_txt}")
print(f"Checkpoint: {ckpt_path} ({ckpt_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Extract the Python source from the log file.
# The log format is: full training script, then training output lines.
# We find where the script ends by looking for the if __name__ block and its body.
raw_lines = log_txt.read_text().splitlines()

# Find the end of the Python source: after `if __name__` block, the first line
# that looks like training output (e.g. "step:" or non-Python) marks the boundary.
in_main = False
source_end = len(raw_lines)
for i, line in enumerate(raw_lines):
    if line.strip().startswith("if __name__"):
        in_main = True
    if in_main and i > 0:
        # Training output lines start with known prefixes or are unindented non-Python
        stripped = line.strip()
        if stripped and not stripped.startswith("#") and ":" in stripped:
            # Check if it looks like a log line (e.g. "step:1/200" or "saved_model:...")
            first_token = stripped.split(":")[0].split("(")[0].split("[")[0]
            if first_token in (
                "step",
                "val_progress",
                "saved_model",
                "serialized_model_int8_zlib",
                "final_int8_zlib_roundtrip",
                "final_int8_zlib_roundtrip_exact",
                "stopping_early",
                "WARNING",
            ):
                source_end = i
                break

script_source = "\n".join(raw_lines[:source_end])
print(f"Extracted {source_end} lines of Python source from {log_txt.name}")

In [ ]:
# Import the training script as a module (without running __main__)
# We strip the if __name__ == "__main__" block to avoid executing training.
main_idx = None
for i, line in enumerate(raw_lines[:source_end]):
    if line.strip().startswith("if __name__"):
        main_idx = i
        break

importable_source = "\n".join(raw_lines[:main_idx]) if main_idx else script_source

# Write to a temp file and import
_tmpdir = tempfile.mkdtemp()
_tmp_path = os.path.join(_tmpdir, f"{RUN_ID.replace('-', '_')}_module.py")
with open(_tmp_path, "w") as f:
    f.write(importable_source)

spec = importlib.util.spec_from_file_location("train_module", _tmp_path)
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)

print(
    f"Imported module with attributes: {[a for a in dir(train_module) if not a.startswith('_')][:20]}"
)

## 2. Instantiate the model and load weights

In [ ]:
# Get hyperparameters from the imported module
hp = train_module.Hyperparameters()
print("Hyperparameters:")
for field in [
    "vocab_size",
    "num_layers",
    "model_dim",
    "num_heads",
    "num_kv_heads",
    "mlp_mult",
    "tie_embeddings",
    "logit_softcap",
    "rope_base",
]:
    if hasattr(hp, field):
        print(f"  {field}: {getattr(hp, field)}")

In [ ]:
# Build the model using the same constructor args from the training script.
# The GPT class signature can vary between records, so we introspect.
import inspect

GPT = train_module.GPT
sig = inspect.signature(GPT.__init__)
init_params = [p for p in sig.parameters if p != "self"]
print(f"GPT.__init__ params: {init_params}")

# Map from common init param names to hyperparameter fields
param_map = {
    "vocab_size": hp.vocab_size,
    "num_layers": hp.num_layers,
    "dim": hp.model_dim,
    "num_heads": hp.num_heads,
    "num_kv_heads": hp.num_kv_heads,
    "mlp_mult": hp.mlp_mult,
    "logit_chunk_tokens": getattr(hp, "logit_chunk_tokens", 0),
    "logit_softcap": getattr(hp, "logit_softcap", 30.0),
    "rope_base": getattr(hp, "rope_base", 10000.0),
    "tied_embed_init_std": getattr(hp, "tied_embed_init_std", 0.005),
    "qk_gain_init": getattr(hp, "qk_gain_init", 1.5),
}

# Build kwargs from what the constructor actually accepts
kwargs = {}
for p in init_params:
    if p in param_map:
        kwargs[p] = param_map[p]
    else:
        # Try getting it from hyperparameters directly
        if hasattr(hp, p):
            kwargs[p] = getattr(hp, p)
        else:
            print(f"  WARNING: unknown init param '{p}' — using default")

print(f"\nConstructing GPT with: {kwargs}")
model = GPT(**kwargs)
print(f"Model created successfully.")

In [ ]:
# Load checkpoint weights
if USE_QUANTIZED:
    with open(ckpt_path, "rb") as f:
        quant_blob = f.read()
    quant_obj = pickle.loads(zlib.decompress(quant_blob))
    flat_state = train_module.dequantize_state_dict_int8(quant_obj)
    print(f"Loaded quantized checkpoint: {len(flat_state)} tensors")
else:
    raw = dict(np.load(str(ckpt_path)))
    flat_state = {k: mx.array(v) for k, v in raw.items()}
    print(f"Loaded raw checkpoint: {len(flat_state)} tensors")

model.update(tree_unflatten(list(flat_state.items())))
mx.eval(model.parameters())

# Count parameters
num_params = sum(v.size for _, v in tree_flatten(model.parameters()))
print(f"Model parameters: {num_params:,}")

## 3. Load tokenizer and data

In [ ]:
# Load tokenizer
sp = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)
print(f"Tokenizer loaded: vocab_size={sp.vocab_size()}")

In [ ]:
# Use the baseline data loading utilities
# We import from train_gpt_mlx.py directly for the data loading parts
sys.path.insert(0, str(Path(".").resolve()))
from train_gpt_mlx import (
    load_data_shard,
    load_validation_tokens,
    TokenStream,
    TokenLoader,
    build_sentencepiece_luts,
)

# Load validation tokens
val_pattern = f"{DATA_PATH}/fineweb_val_*.bin"
val_tokens = load_validation_tokens(val_pattern, seq_len=SEQ_LEN)
print(
    f"Validation tokens: {val_tokens.shape[0]:,} (enough for {(val_tokens.shape[0] - 1) // SEQ_LEN} sequences)"
)
n_documents = np.sum(val_tokens == 1)
print(
    f"Found {n_documents} documents in validation set (start tokens). Average document length: {val_tokens.shape[0] / n_documents:.1f} tokens."
)

# Set up train data loader
train_pattern = f"{DATA_PATH}/fineweb_train_*.bin"
train_loader = TokenLoader(train_pattern, dataset_name="train")
print(f"Train loader ready.")

# Build byte counting LUTs for BPB calculation
base_bytes_lut, has_leading_space_lut, is_boundary_token_lut = build_sentencepiece_luts(
    sp, hp.vocab_size
)

In [ ]:
from matplotlib import pyplot as plt
from numpy import mean

# indices of start tokens
start_token_indices = np.where(val_tokens == 1)[0]

# length of each document (distance between start tokens)
doc_lengths = np.diff(start_token_indices, append=len(val_tokens))
mean_doc_length = mean(doc_lengths)
print(f"Mean document length: {mean_doc_length:.1f} tokens")

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(doc_lengths, bins=100, color="skyblue", edgecolor="black")
ax.set_xlabel("Document Length (tokens)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Document Lengths in Validation Set")
ax.set_yscale("log")
for quantile in [0.5, 0.9, 0.99, 0.999, 0.9999]:
    color = "0.5"
    q_value = np.quantile(doc_lengths, quantile)
    ax.vlines(q_value, ymin=1, ymax=ax.get_ylim()[1], colors=color, linestyles="dashed")
    ax.text(
        q_value,
        ax.get_ylim()[1] * 0.9,
        f"{int(quantile * 100)}%: {q_value:.0f}",
        rotation=90,
        verticalalignment="top",
        color=color,
    )

# plot sequence of document lengths
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(doc_lengths, color="blue")
ax.set_xlabel("Document Index")
ax.set_ylabel("Document Length (tokens)")
ax.set_title("Document Lengths in Validation Set")

# cumulative total tokens covered
sorted_lengths = np.sort(doc_lengths)
cumulative_tokens = np.cumsum(sorted_lengths)
total_tokens = cumulative_tokens[-1]
cumulative_coverage = cumulative_tokens / total_tokens

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sorted_lengths, cumulative_coverage, color="blue")
ax.set_xlabel("Document Length (tokens)")
ax.set_ylabel("Cumulative Coverage of Validation Tokens")
ax.set_title("Cumulative Coverage of Validation Tokens by Document Length")
ax.grid(True)
coverages = {}
for coverage in [0.2, 0.3, 0.5, 0.9, 0.95]:
    color = "0.8"
    idx = np.searchsorted(cumulative_coverage, coverage)
    length_at_coverage = sorted_lengths[idx]
    coverages[coverage] = length_at_coverage
    ax.vlines(
        length_at_coverage,
        ymin=0,
        ymax=coverage,
        colors=color,
        linestyles="dashed",
    )
    ax.hlines(
        coverage,
        xmin=0,
        xmax=length_at_coverage,
        colors=color,
        linestyles="dashed",
    )
    ax.text(
        length_at_coverage,
        coverage,
        f"{int(coverage * 100)}%: {length_at_coverage:.0f} tokens",
        rotation=90,
        verticalalignment="bottom",
        color="0.5",
    )


# text box with coverage info
textstr = "\n".join(
    [
        f"{int(cov * 100)}% coverage: {length} tokens"
        for cov, length in coverages.items()
    ]
)
props = dict(boxstyle="round", facecolor="white", alpha=0.8)
ax.text(
    0.95,
    0.05,
    textstr,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=props,
)
ax.set_xscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# Factual recall / knowledge token analysis
# Per document (BOS token id=1), find knowledge-bearing spans (numbers, proper nouns)
# in the decoded text, map them back to token counts, and check if they repeat in the doc.
import re
from collections import Counter

BOS_ID = 1
pieces = [sp.id_to_piece(int(tid)) for tid in range(sp.vocab_size())]

# Build a mapping: for each token, how many decoded characters does it produce?
# SentencePiece ▁ = one space character
piece_lens = []
for p in pieces:
    piece_lens.append(len(p.replace("▁", " ")))

# Patterns to find in decoded text
# Numbers: sequences of digits, possibly with internal commas/periods (e.g. 1,000 or 3.14)
number_re = re.compile(r"\b\d[\d,]*\.?\d*\b")
# Proper nouns: capitalized words not at sentence start.
# We find all capitalized words, then filter out sentence-initial ones.
# A proper noun: uppercase letter followed by lowercase letters, possibly hyphenated/multi-word
name_re = re.compile(r"\b[A-Z][a-z]+(?:[\s'-][A-Z][a-z]+)*\b")
# Sentence boundary: .!?:" followed by space (to detect sentence-initial caps)
sent_end_re = re.compile(r"[.!?:]\s*$")

# Common words that are often capitalized but aren't proper nouns
COMMON_CAPS = {
    "The",
    "This",
    "That",
    "These",
    "Those",
    "There",
    "Their",
    "They",
    "Then",
    "When",
    "Where",
    "What",
    "Which",
    "While",
    "Who",
    "Why",
    "How",
    "Here",
    "His",
    "Her",
    "He",
    "She",
    "Its",
    "Our",
    "Your",
    "My",
    "But",
    "And",
    "For",
    "Not",
    "Are",
    "Was",
    "Were",
    "Has",
    "Had",
    "Have",
    "Will",
    "Would",
    "Could",
    "Should",
    "May",
    "Can",
    "Did",
    "Does",
    "Been",
    "Being",
    "With",
    "From",
    "Into",
    "Over",
    "After",
    "Before",
    "Between",
    "Under",
    "About",
    "Some",
    "Many",
    "Most",
    "Other",
    "Each",
    "Every",
    "Both",
    "Such",
    "Just",
    "Also",
    "Now",
    "Still",
    "Even",
    "Only",
    "Very",
    "More",
    "Much",
    "Any",
    "All",
    "One",
    "Two",
    "Three",
    "Four",
    "Five",
    "Six",
    "Seven",
    "Eight",
    "Nine",
    "Ten",
    "First",
    "Last",
    "New",
    "Old",
    "Good",
    "Great",
    "Big",
    "Small",
    "Long",
    "High",
    "Low",
    "Few",
    "Less",
    "Like",
    "Well",
    "Way",
    "Day",
    "Time",
    "Year",
    "People",
    "Part",
    "Place",
    "Case",
    "Week",
    "Company",
    "System",
    "Program",
    "Question",
    "Work",
    "Government",
    "Number",
    "Night",
    "Point",
    "Home",
    "Water",
    "Room",
    "Mother",
    "Area",
    "Money",
    "Story",
    "Fact",
    "Month",
    "Lot",
    "Right",
    "Study",
    "Book",
    "Eye",
    "Job",
    "Word",
    "Business",
    "Issue",
    "Side",
    "Kind",
    "Head",
    "House",
    "Service",
    "Friend",
    "Father",
    "Power",
    "Hour",
    "Game",
    "Line",
    "End",
    "Members",
    "Family",
    "Back",
    "According",
    "However",
    "Although",
    "Because",
    "Since",
    "Whether",
    "Another",
    "During",
    "Without",
    "Something",
    "Everything",
    "Nothing",
    "Perhaps",
    "Instead",
    "Despite",
    "Rather",
    "Together",
    "Already",
    "Probably",
    "Often",
    "Sometimes",
    "Never",
    "Always",
    "If",
    "It",
    "In",
    "Is",
    "So",
    "Or",
    "As",
    "At",
    "By",
    "No",
    "We",
    "An",
    "On",
    "Do",
    "Up",
}

doc_starts = np.where(val_tokens == BOS_ID)[0]
doc_boundaries = np.concatenate([doc_starts, [len(val_tokens)]])

total_tokens = 0
num_tok = 0
name_tok = 0
num_facts = 0
name_facts = 0
num_facts_first = 0
num_facts_repeat = 0
name_facts_first = 0
name_facts_repeat = 0
num_tok_first = 0
num_tok_repeat = 0
name_tok_first = 0
name_tok_repeat = 0

examples = []
MAX_EXAMPLES = 2000
CTX_CHARS = 40

for di in range(len(doc_boundaries) - 1):
    start, end = int(doc_boundaries[di]), int(doc_boundaries[di + 1])
    doc_toks = val_tokens[start:end]
    doc_len = len(doc_toks)
    if doc_len < 2:
        continue
    total_tokens += doc_len

    # Decode document and build char->token mapping
    char_to_tok = []  # for each character in decoded text, which token index produced it
    decoded_chars = []
    for pos in range(doc_len):
        tid = int(doc_toks[pos])
        p = pieces[tid] if tid < len(pieces) else ""
        text = p.replace("▁", " ")
        for ch in text:
            decoded_chars.append(ch)
            char_to_tok.append(pos)
    decoded_text = "".join(decoded_chars)

    # Find all facts in decoded text
    facts_in_doc = []  # (char_start, char_end, text, type)

    # Numbers
    for m in number_re.finditer(decoded_text):
        facts_in_doc.append((m.start(), m.end(), m.group(), "num"))

    # Names: capitalized words, filtering out sentence-initial and common words
    for m in name_re.finditer(decoded_text):
        name = m.group()
        # Skip common words
        first_word = name.split()[0].split("'")[0].split("-")[0]
        if first_word in COMMON_CAPS:
            continue
        # Skip if at very start of doc (first 3 chars)
        if m.start() < 3:
            continue
        # Skip if preceded by sentence-ending punctuation
        prefix = decoded_text[max(0, m.start() - 3) : m.start()]
        if sent_end_re.search(prefix):
            continue
        facts_in_doc.append((m.start(), m.end(), name, "name"))

    # For each fact: count tokens it spans, check if text appeared earlier
    seen_texts = set()
    for cs, ce, text, kind in sorted(facts_in_doc):
        # Count unique token indices spanned
        tok_indices = set(char_to_tok[cs:ce])
        n_toks = len(tok_indices)

        is_first = text not in seen_texts
        seen_texts.add(text)

        if kind == "num":
            num_facts += 1
            num_tok += n_toks
            if is_first:
                num_facts_first += 1
                num_tok_first += n_toks
            else:
                num_facts_repeat += 1
                num_tok_repeat += n_toks
        else:
            name_facts += 1
            name_tok += n_toks
            if is_first:
                name_facts_first += 1
                name_tok_first += n_toks
            else:
                name_facts_repeat += 1
                name_tok_repeat += n_toks

        if len(examples) < MAX_EXAMPLES:
            ctx_b = decoded_text[max(0, cs - CTX_CHARS) : cs]
            ctx_a = decoded_text[ce : ce + CTX_CHARS]
            examples.append(
                {
                    "type": kind,
                    "fact": text,
                    "n_tokens": n_toks,
                    "first_occ": is_first,
                    "context": f"...{ctx_b}[{text}]{ctx_a}...",
                    "doc_idx": di,
                }
            )

print(f"Total tokens: {total_tokens:,}")
print(f"Documents: {len(doc_boundaries) - 1:,}")
print()
print("=== Numbers ===")
print(
    f"  Facts: {num_facts:,}  |  Tokens: {num_tok:,} ({100 * num_tok / total_tokens:.2f}%)"
)
print(f"  Avg tokens/fact: {num_tok / max(num_facts, 1):.2f}")
print(
    f"  First occ: {num_facts_first:,} facts, {num_tok_first:,} tok  |  Repeated: {num_facts_repeat:,} facts, {num_tok_repeat:,} tok"
)
print()
print("=== Names (proper nouns) ===")
print(
    f"  Facts: {name_facts:,}  |  Tokens: {name_tok:,} ({100 * name_tok / total_tokens:.2f}%)"
)
print(f"  Avg tokens/fact: {name_tok / max(name_facts, 1):.2f}")
print(
    f"  First occ: {name_facts_first:,} facts, {name_tok_first:,} tok  |  Repeated: {name_facts_repeat:,} facts, {name_tok_repeat:,} tok"
)
print()
tk = num_tok + name_tok
tf = num_facts + name_facts
tf_first = num_tok_first + name_tok_first
tf_repeat = num_tok_repeat + name_tok_repeat
print(f"=== Combined ===")
print(
    f"  Knowledge tokens: {tk:,} ({100 * tk / total_tokens:.2f}%) in {tf:,} facts ({tk / max(tf, 1):.2f} tok/fact)"
)
print(f"  Factual recall: {tf_first:,} tok ({100 * tf_first / total_tokens:.2f}%)")
print(
    f"  Context-predictable: {tf_repeat:,} tok ({100 * tf_repeat / total_tokens:.2f}%)"
)

# Save examples
import json as _json

with open("knowledge_token_examples.jsonl", "w") as f:
    for ex in examples:
        f.write(_json.dumps(ex) + "\n")
print(f"\nSaved {len(examples)} examples to knowledge_token_examples.jsonl")

# Show samples
for label, kind in [("number", "num"), ("name", "name")]:
    print(f"\n=== Sample {label} facts ===")
    for ex in [e for e in examples if e["type"] == kind][:15]:
        tag = "FIRST" if ex["first_occ"] else "REPEAT"
        print(f"  [{tag}] ({ex['n_tokens']}tok) {ex['context']}")

## 4. Run the model on examples

In [ ]:
def get_val_batch(batch_idx: int = 0, seq_len: int = SEQ_LEN):
    """Get a batch from validation data by index."""
    start = batch_idx * seq_len
    end = start + seq_len + 1
    if end > val_tokens.shape[0]:
        raise IndexError(f"batch_idx {batch_idx} out of range")
    chunk = val_tokens[start:end]
    x = mx.array(chunk[:-1].reshape(1, seq_len), dtype=mx.int32)
    y = mx.array(chunk[1:].reshape(1, seq_len), dtype=mx.int32)
    return x, y


def get_train_batch(num_tokens: int = SEQ_LEN, seq_len: int = SEQ_LEN):
    """Get the next batch from the training stream."""
    return train_loader.next_batch(num_tokens, seq_len)


def decode_tokens(token_ids):
    """Decode token IDs to text."""
    if isinstance(token_ids, mx.array):
        token_ids = token_ids.tolist()
    if isinstance(token_ids, np.ndarray):
        token_ids = token_ids.tolist()
    # Flatten if needed
    if isinstance(token_ids[0], list):
        token_ids = token_ids[0]
    return sp.decode(token_ids)


def compute_loss(x, y):
    """Compute cross-entropy loss for a single (x, y) pair."""
    loss = model.loss(x, y)
    mx.eval(loss)
    return float(loss)


print(
    "Helper functions defined: get_val_batch, get_train_batch, decode_tokens, compute_loss"
)

In [ ]:
# === Example: look at a validation sequence and model predictions ===
x, y = get_val_batch(0)

# Compute loss
loss = compute_loss(x, y)
print(f"Loss on val batch 0: {loss:.4f}")
print(f"Perplexity: {np.exp(loss):.2f}")
print()

# Show the input text (first 500 chars)
input_text = decode_tokens(x)
print("=== Input text (first 500 chars) ===")
print(input_text[:500])
print("...")

In [ ]:
# === Example: get model predictions and compare with ground truth ===
x, y = get_val_batch(0)

# Forward pass to get hidden states, then compute logits
hidden = model(x)  # (1, seq_len, dim)
# Logits via tied embeddings
logits = (
    hidden.reshape(-1, model.tok_emb.weight.shape[1])
    @ model.tok_emb.weight.astype(hidden.dtype).T
)
if hasattr(model, "softcap"):
    logits = model.softcap(logits)
mx.eval(logits)

# Get top-k predictions for each position
probs = mx.softmax(logits.astype(mx.float32), axis=-1)
mx.eval(probs)

# Show predictions for a few positions
K = 5
positions = [0, 10, 50, 100, 200]
y_flat = y.reshape(-1)
print(f"Length of sequence: {y_flat.shape[0]}")

for pos in positions:
    if pos >= logits.shape[0]:
        break
    pos_probs = np.array(probs[pos])
    top_k_ids = np.argsort(pos_probs)[-K:][::-1]
    actual_id = int(y_flat[pos])
    actual_prob = float(pos_probs[actual_id])

    print(f"\n--- Position {pos} ---")
    print(f"  Context: ...{decode_tokens(x[0, max(0, pos - 5) : pos + 1].tolist())}")
    print(
        f"  Actual next: '{sp.id_to_piece(actual_id)}' (id={actual_id}, prob={actual_prob:.4f})"
    )
    print(f"  Top-{K} predictions:")
    for rank, tid in enumerate(top_k_ids):
        tid = int(tid)
        print(
            f"    {rank + 1}. '{sp.id_to_piece(tid)}' (id={tid}, prob={float(pos_probs[tid]):.4f})"
            + (" <<<" if tid == actual_id else "")
        )

In [ ]:
# === Example: greedy autoregressive generation from a prompt ===
def generate(prompt_tokens, max_new_tokens=100, temperature=0.8, top_k=50):
    """Simple autoregressive generation."""
    if isinstance(prompt_tokens, list):
        tokens = list(prompt_tokens)
    else:
        tokens = (
            prompt_tokens.tolist()
            if hasattr(prompt_tokens, "tolist")
            else list(prompt_tokens)
        )

    for _ in range(max_new_tokens):
        # Use last SEQ_LEN tokens as context
        ctx = tokens[-SEQ_LEN:]
        x = mx.array([ctx], dtype=mx.int32)
        hidden = model(x)
        # Get logits for the last position
        last_hidden = hidden[0, -1, :]  # (dim,)
        logits = last_hidden @ model.tok_emb.weight.astype(hidden.dtype).T
        if hasattr(model, "softcap"):
            logits = model.softcap(logits)
        logits = logits.astype(mx.float32)

        if temperature <= 0:
            # Greedy
            next_id = int(mx.argmax(logits))
        else:
            # Top-k sampling
            logits_np = np.array(logits)
            if top_k > 0 and top_k < logits_np.shape[0]:
                indices_to_remove = logits_np < np.partition(logits_np, -top_k)[-top_k]
                logits_np[indices_to_remove] = -float("inf")
            logits_np = logits_np / temperature
            logits_np -= logits_np.max()
            probs = np.exp(logits_np)
            probs /= probs.sum()
            next_id = int(np.random.choice(len(probs), p=probs))

        tokens.append(next_id)

    return tokens


# Generate from the start of a validation sequence
x, y = get_val_batch(5)
prompt_len = 50  # Use first 50 tokens as prompt
prompt = x[0, :prompt_len].tolist()

print("=== Prompt ===")
print(decode_tokens(prompt))
print()

generated = generate(prompt, max_new_tokens=100, temperature=0.8)
print("=== Generated continuation ===")
print(decode_tokens(generated[prompt_len:]))
print()

print("=== Actual continuation ===")
print(decode_tokens(y[0, prompt_len - 1 : prompt_len + 99].tolist()))

In [ ]:
# === Example: compare loss on a few train vs val batches ===
print("Validation losses:")
for i in range(5):
    x, y = get_val_batch(i)
    loss = compute_loss(x, y)
    print(f"  val batch {i}: loss={loss:.4f}  ppl={np.exp(loss):.2f}")

print("\nTraining losses:")
for i in range(5):
    x, y = get_train_batch()
    loss = compute_loss(x, y)
    print(f"  train batch {i}: loss={loss:.4f}  ppl={np.exp(loss):.2f}")

## 5. Parse training logs and plot training curves

Parse `step:` lines from `models/*/log.txt` into pandas DataFrames and plot loss curves.

In [ ]:
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def parse_log(path: str | Path) -> pd.DataFrame:
    """Parse a training log file into a DataFrame.

    Extracts all `step:` lines and parses key:value pairs into columns.
    Handles both train lines (step, train_loss, train_time, step_avg, tok_s)
    and val lines (step, val_loss, val_bpb, train_time, step_avg).

    Multiple runs appended to the same file are detected by step resets
    (current step <= previous step) and tagged with a `run` column (0-indexed).

    Returns a DataFrame with one row per `step:` line.
    """
    path = Path(path)
    rows = []
    pattern = re.compile(r"(\w+):([\w./+-]+)")
    unit_suffixes = ("ms", "s", "x")
    run_idx = 0
    prev_step = -1

    for line in path.read_text().splitlines():
        stripped = line.strip()
        if not stripped.startswith("step:"):
            continue
        fields = dict(pattern.findall(stripped))
        if not fields:
            continue

        row = {}
        for k, v in fields.items():
            if k == "step" and "/" in v:
                parts = v.split("/")
                row["step"] = int(parts[0])
                row["total_steps"] = int(parts[1])
            else:
                v_clean = v
                for suffix in unit_suffixes:
                    if v_clean.endswith(suffix) and len(v_clean) > len(suffix):
                        v_clean = v_clean[: -len(suffix)]
                        break
                try:
                    row[k] = int(v_clean)
                except ValueError:
                    try:
                        row[k] = float(v_clean)
                    except ValueError:
                        row[k] = v

        # Detect run boundary: step resets to a value <= previous step
        cur_step = row.get("step", prev_step + 1)
        if cur_step <= prev_step and prev_step > 0:
            run_idx += 1
        prev_step = cur_step
        row["run"] = run_idx
        rows.append(row)

    df = pd.DataFrame(rows)
    if "train_time" in df.columns:
        df["train_time_min"] = df["train_time"] / 60_000.0
    return df


def split_train_val(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split a parsed log DataFrame into train and val DataFrames."""
    is_val = (
        df["val_loss"].notna()
        if "val_loss" in df.columns
        else pd.Series(False, index=df.index)
    )
    return df[~is_val].copy(), df[is_val].copy()


def get_runs(df: pd.DataFrame) -> dict[int, pd.DataFrame]:
    """Split a parsed log DataFrame by run index."""
    return {run: group for run, group in df.groupby("run")}


# Parse all log files
log_dir = Path("models")
logs = {}
for f in sorted(log_dir.glob("*/log.txt")):
    name = f.parent.name
    df = parse_log(f)
    if len(df) > 0:
        logs[name] = df
        n_runs = df["run"].nunique()
        train, val = split_train_val(df)
        print(
            f"{name}: {len(df)} step lines, {n_runs} run(s) ({len(train)} train, {len(val)} val)"
        )

print(f"\nParsed {len(logs)} log files. Access via logs['name'].")
print("Use get_runs(df) to split into per-run DataFrames.")

In [ ]:
# Quick look at one DataFrame
sample_name = list(logs.keys())[0]
print(f"Sample: logs['{sample_name}']")
logs[sample_name].head(10)

In [ ]:
# --- Training loss vs step (each run as a separate line) ---
fig, ax = plt.subplots(figsize=(12, 5))
for name, df in logs.items():
    for run_idx, run_df in get_runs(df).items():
        train, _ = split_train_val(run_df)
        if len(train) == 0 or "train_loss" not in train.columns:
            continue
        t = train.dropna(subset=["train_loss"])
        n_runs = df["run"].nunique()
        label = f"{name}" if n_runs == 1 else f"{name}/run{run_idx}"
        ax.plot(t["step"], t["train_loss"], label=label, alpha=0.8, linewidth=1)

ax.set_xlabel("Step")
ax.set_ylabel("Train Loss")
ax.set_title("Training Loss vs Step")
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Latest run per file: smoothed training loss vs step ---
EMA_ALPHA = 0.05  # smaller = smoother (0.05 ≈ ~20-step window)

fig, ax = plt.subplots(figsize=(12, 5))
for name, df in logs.items():
    last_run = df["run"].max()
    run_df = df[df["run"] == last_run]
    train, _ = split_train_val(run_df)
    if len(train) == 0 or "train_loss" not in train.columns:
        continue
    t = train.dropna(subset=["train_loss"])
    if len(t) < 2:
        continue
    smoothed = t["train_loss"].ewm(alpha=EMA_ALPHA, adjust=False).mean()
    ax.plot(t["step"].values, smoothed.values, label=name, alpha=0.9, linewidth=1.5)

ax.set_xlabel("Step")
ax.set_ylabel("Train Loss (EMA smoothed)")
ax.set_title("Latest Run per Log — Training Loss vs Step")
ax.legend(loc="upper right", fontsize=8)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from itertools import cycle

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_cycle = cycle(colors)


In [ ]:
# --- Training loss vs step (smoothed with EMA) ---
EMA_ALPHA = 0.01  # smaller = smoother (0.05 ≈ ~20-step window)

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, df) in enumerate(logs.items()):
    color = next(color_cycle)
    for run_idx, run_df in get_runs(df).items():
        train, _ = split_train_val(run_df)
        if len(train) == 0 or "train_loss" not in train.columns:
            continue
        t = train.dropna(subset=["train_loss"])
        if len(t) < 1000:
            continue
        smoothed = t["train_loss"].ewm(alpha=EMA_ALPHA, adjust=False).mean()
        n_runs = df["run"].nunique()
        label = f"{name}" if n_runs == 1 else f"{name}/run{run_idx}"
        ax.plot(
            t["step"].values,
            smoothed.values,
            label=label,
            alpha=0.9,
            linewidth=1.5,
            color=color,
        )

ax.set_xlabel("Step")
ax.set_ylabel("Train Loss (EMA smoothed)")
ax.set_title(f"Training Loss vs Step (EMA alpha={EMA_ALPHA})")
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.set_ylim(2, 4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Training loss vs wall-clock time (minutes), per run ---
fig, ax = plt.subplots(figsize=(12, 5))
for name, df in logs.items():
    for run_idx, run_df in get_runs(df).items():
        train, _ = split_train_val(run_df)
        if (
            len(train) == 0
            or "train_loss" not in train.columns
            or "train_time_min" not in train.columns
        ):
            continue
        t = train.dropna(subset=["train_loss"])
        n_runs = df["run"].nunique()
        label = f"{name}" if n_runs == 1 else f"{name}/run{run_idx}"
        ax.plot(
            t["train_time_min"], t["train_loss"], label=label, alpha=0.8, linewidth=1
        )

ax.set_xlabel("Wall-clock Time (minutes)")
ax.set_ylabel("Train Loss")
ax.set_title("Training Loss vs Time")
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Validation BPB vs time (if available), per run ---
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
has_val = False
fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, df) in enumerate(logs.items()):
    color = color_cycle[i]
    for run_idx, run_df in get_runs(df).items():
        _, val = split_train_val(run_df)
        if len(val) == 0 or "val_bpb" not in val.columns:
            continue
        v = val.dropna(subset=["val_bpb"])
        if len(v) == 0:
            continue
        has_val = True
        n_runs = df["run"].nunique()
        label = f"{name}" if n_runs == 1 else f"{name}/run{run_idx}"
        ax.plot(
            v["train_time_min"],
            v["val_bpb"],
            "o-",
            label=label,
            markersize=4,
            color=color,
        )

if has_val:
    ax.set_xlabel("Wall-clock Time (minutes)")
    ax.set_ylabel("Validation BPB")
    ax.set_title("Validation BPB vs Time")
    ax.legend(loc="upper right", fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    plt.close()
    print("No validation BPB data found in any log.")

In [ ]:
# --- Throughput (tok/s) vs step, per run ---
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in logs.items():
    for run_idx, run_df in get_runs(df).items():
        train, _ = split_train_val(run_df)
        if len(train) == 0 or "tok_s" not in train.columns:
            continue
        t = train.dropna(subset=["tok_s"])
        n_runs = df["run"].nunique()
        label = f"{name}" if n_runs == 1 else f"{name}/run{run_idx}"
        ax.plot(t["step"], t["tok_s"], label=label, alpha=0.7, linewidth=1)

ax.set_xlabel("Step")
ax.set_ylabel("Tokens/sec")
ax.set_title("Training Throughput")
ax.legend(loc="lower right", fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()